In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_boston

# Data Preprocessing

In [3]:
boston = load_boston()
X = pd.DataFrame(boston.data, columns = boston.feature_names)
y = pd.Series(boston.target, name = 'MEDV')

C:\Users\Enass Irfan\anaconda3\lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function load_boston is deprecated; `load_boston` is deprecated in 1.0 and will be removed in 1.2.

    The Boston housing prices dataset has an ethical problem. You can refer to
    the documentation of this function for further details.

    The scikit-learn maintainers therefore strongly discourage the use of this
    dataset unless the purpose of the code is to study and educate about
    ethical issues in data science and machine learning.

    In this special case, you can fetch the dataset from the original
    source::

        import pandas as pd
        import numpy as np


        data_url = "http://lib.stat.cmu.edu/datasets/boston"
        raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
        data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
        target = raw_df.values[1::2, 2]

    Alternative datasets include the California housing dat

In [4]:
from sklearn.model_selection import train_test_split
X_normalized = (X - X.mean()) / X.std()

In [5]:

X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size = 0.2, random_state = 42)

# Model Implementation

Linear Regression

In [6]:
class LinearRegression:
    def __init__(self, lr = 0.01, n_iters = 1000):
        self.lr = lr
        self.n_iters = n_iters
    
    def fit(self, X, y):
        self.m, self.n = X.shape
        self.weights = np.zeros(self.n)
        self.bias = 0
        
        for _ in range(self.n_iters):
            y_pred = self.predict(X)
            dw = -(2/self.m) * np.dot(X.T, (y- y_pred))
            db = -(2/self.m) * np.sum(y - y_pred)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

Random Forest

In [7]:
from collections import Counter
import random

class DecisionTreeRegressor:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

        # Initialize attributes
        self.feature_index = None
        self.threshold = None
        self.left = None
        self.right = None
        self.value = None

    def fit(self, X, y, depth=0):
        self.n_samples, self.n_features = X.shape
        self.value = np.mean(y)

        # Stop conditions
        if depth >= self.max_depth or self.n_samples < self.min_samples_split:
            return

        best_mse = float('inf')
        for feature in range(self.n_features):
            thresholds = np.unique(X[:, feature])
            for t in thresholds:
                left_idx = X[:, feature] <= t
                right_idx = X[:, feature] > t
                if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                    continue
                y_left, y_right = y[left_idx], y[right_idx]
                mse = np.mean((y_left - y_left.mean())**2) + np.mean((y_right - y_right.mean())**2)

                if mse < best_mse:
                    best_mse = mse
                    self.feature_index = feature
                    self.threshold = t
                    best_left_idx = left_idx
                    best_right_idx = right_idx

        if self.feature_index is not None:
            self.left = DecisionTreeRegressor(self.max_depth, self.min_samples_split)
            self.right = DecisionTreeRegressor(self.max_depth, self.min_samples_split)
            self.left.fit(X[best_left_idx], y[best_left_idx], depth + 1)
            self.right.fit(X[best_right_idx], y[best_right_idx], depth + 1)

    def predict(self, X):
        if self.feature_index is None:
            return np.full(X.shape[0], self.value)

        left_idx = X[:, self.feature_index] <= self.threshold
        right_idx = X[:, self.feature_index] > self.threshold
        
        y_pred = np.empty(X.shape[0])
        y_pred[left_idx] = self.left.predict(X[left_idx])
        y_pred[right_idx] = self.right.predict(X[right_idx])
        return y_pred

class RandomForestRegressor:
    def __init__(self, n_trees = 10, max_depth = 10):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []
    
    def fit(self, X , y):
        self.trees = []
        for _ in range(self.n_trees):
            indices = np.random.choice(len(X), len(X), replace = True)
            X_sample, y_sample = X[indices], y[indices]
            tree = DecisionTreeRegressor(max_depth = self.max_depth)
            self.trees.append(tree)
    
    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        return np.mean(tree_preds, axis = 0)

XGBoost

In [8]:
class XGBoostRegressor:
    def __init__(self, n_estimators = 50, learning_rate = 0.1, max_depth = 3):  
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
    
    def fit(self, X, y):
        pred = np,zeros(len(y))
        for _ in range(self.n_estimators):
            residuals = y - pred
            tree = DecisionTreeRegressor(max_depth = self.max_depth)
            tree.fit(X, residuals)
            pred += self.learning_rate * tree.predict(X)
            self.trees.append(tree)
    
    def predict(self, X):
        pred = np.zeros(X.shape[0])
        for tree in self.trees:
            pred += self.learning_rate * tree.predict(X)
        return pred

# Performance Comparison

In [9]:
from sklearn.metrics import mean_squared_error, r2_score

def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return rmse, r2

# Feature Importance

In [10]:
def extract_feature_importance(tree, importance):
    # Base case: if the tree is a leaf node, stop recursion
    if not hasattr(tree, 'feature_index') or tree.feature_index is None:
        return
    # Count the split
    importance[tree.feature_index] += 1
    # Recursively traverse left and right children
    if tree.left:
        extract_feature_importance(tree.left, importance)
    if tree.right:
        extract_feature_importance(tree.right, importance)

def plot_feature_importance(model, feature_names):
    importance = np.zeros(len(feature_names))
    for tree in model.trees:
        extract_feature_importance(tree, importance)

    # Plotting
    plt.barh(feature_names, importance)
    plt.xlabel("Feature Importance (Split Count)")
    plt.title("Feature Importance")
    plt.show()


In [11]:
# Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train.values, y_train.values)
y_pred_lr = lr_model.predict(X_test.values)
print("Linear Regression:", evaluate(y_test, y_pred_lr))

rf_model = RandomForestRegressor()
rf_model.fit(X_train.values, y_train.values)
y_pred_rf = rf_model.predict(X_test.values)
print("Random Forest:", evaluate(y_test, y_pred_rf))
plot_feature_importance(rf_model, X.columns)

# XGBoost
xgb_model = XGBoostRegressor()
xgb_model.fit(X_train.values, y_train.values)
y_pred_xgb = xgb_model.predict(X_test.values)
print("XGBoost:", evaluate(y_test, y_pred_xgb))
plot_feature_importance(xgb_model, X.columns)


Linear Regression: (4.971153264273616, 0.6630152746535407)


TypeError: unsupported operand type(s) for +: 'NoneType' and 'NoneType'